# Cleaning Up Books — extra (full dump)

Same pipeline as `book_data.ipynb`, run on `goodreads_books_extra.parquet`.
Writes `editions_extra.parquet` and `english_works_extra.parquet`.

The edition-level file is the one that matters for the later merge: works must be
re-aggregated across both runs, not merged after the fact.

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter
from tqdm import tqdm
from langdetect import detect, DetectorFactory, LangDetectException

In [4]:
RAW     = Path('../data/initial')
INTERIM = Path('../data/interim')
INTERIM.mkdir(parents=True, exist_ok=True)

SRC = RAW / 'goodreads_books_extra.parquet'

## Load

One file, not eight. `genre` is already a list from the corpus-wide genre map,
so there is no filename to read it from.

In [5]:
df = pd.read_parquet(SRC)
print(f"{len(df):,} rows, {len(df.columns)} cols")

# genre came from goodreads_book_genres_initial, already a list per book
df['genre'] = df['genre'].apply(lambda v: list(v) if v is not None else [])
print(f"books with no genre: {df.genre.apply(len).eq(0).sum():,}")
df.head()

792,133 rows, 30 cols
books with no genre: 244,301


,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series,genre
0,0743509986,6,[],US,,"[{'count': '160', 'name': 'fiction'}, {'count'...",,false,3.23,B000FC0PBC,...,Abridged,2001,https://www.goodreads.com/book/show/1333909.Go...,https://s.gr-assets.com/assets/nophoto/book/11...,1333909,10,1323437,Good Harbor,Good Harbor,"[biography, fiction, historical fiction, history]"
1,0743294297,3282,[],US,eng,"[{'count': '728', 'name': 'chick-lit'}, {'coun...",,false,3.49,B002ENBLOK,...,,2009,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184,6243154,Best Friends Forever,Best Friends Forever,"[crime, fiction, mystery, romance, thriller]"
2,0922915113,39,[],US,,"[{'count': '22', 'name': 'occult'}, {'count': ...",,false,3.81,B00AFYVB8Q,...,,2000,https://www.goodreads.com/book/show/287149.The...,https://images.gr-assets.com/books/1328768789m...,287149,986,278586,The Devil's Notebook,The Devil's Notebook,"[biography, historical fiction, history, non-f..."
3,,3,[],US,ger,"[{'count': '1', 'name': 'library-books-i-read'}]",,false,2.22,,...,,2016,https://www.goodreads.com/book/show/28575155-s...,https://images.gr-assets.com/books/1452886176m...,28575155,9,48735929,Spirit Lake - Die Legende des Wendigo,Spirit Lake - Die Legende des Wendigo,[]
4,0800759494,2885,[],US,,"[{'count': '340', 'name': 'non-fiction'}, {'co...",,false,3.91,B00B853QPM,...,,,https://www.goodreads.com/book/show/89375.90_M...,https://s.gr-assets.com/assets/nophoto/book/11...,89375,68157,2957021,90 Minutes in Heaven: A True Story of Death an...,90 Minutes in Heaven: A True Story of Death an...,"[biography, comics, fiction, graphic, historic..."


In [6]:
for c in ["ratings_count", "text_reviews_count", "average_rating",
          "num_pages", "publication_year", "publication_month", "publication_day"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df[["ratings_count", "average_rating", "num_pages", "publication_year"]].dtypes)

ratings_count         int64
average_rating      float64
num_pages           float64
publication_year    float64
dtype: object


## Tags

In [7]:
def split_shelves(shelves):
    if shelves is None or len(shelves) == 0:
        return [], []
    keep = [(str(s["name"]).strip().lower(), int(s["count"])) for s in shelves]
    return [t for t, _ in keep], [n for _, n in keep]

In [8]:
pairs = df.popular_shelves.map(split_shelves)
df["tags"]       = pairs.str[0]
df["tag_counts"] = pairs.str[1]
print(f"rows with >=1 tag: {df.tags.apply(len).gt(0).mean():.1%}")

rows with >=1 tag: 91.9%


## Collapsing duplicate book_id

The full dump has one record per book_id, so unlike the eight subsets there
should be no cross-file duplication here. Asserted rather than assumed.

In [9]:
print(f"rows            {len(df):,}")
print(f"unique book_id  {df.book_id.nunique():,}")
print(f"duplicate rows  {df.book_id.duplicated().sum():,}")

books = df.drop_duplicates("book_id").reset_index(drop=True)
books["genres"] = books["genre"]
print(f"\n{len(books):,} unique books")

rows            792,133
unique book_id  792,133
duplicate rows  0

792,133 unique books


## Coverage

In [10]:
def filled(s):
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        return s.fillna("").astype(str).str.strip().str.len().gt(0).mean()
    return s.notna().mean()

cols = ["description", "title", "isbn13", "isbn", "asin", "num_pages",
        "publication_year", "language_code", "average_rating", "url", "image_url"]

cov = pd.Series({c: filled(books[c]) for c in cols})
cov["tags"] = books.tags.apply(len).gt(0).mean()
cov.sort_values().map("{:.1%}".format)

asin                 14.1%
language_code        55.7%
isbn                 61.7%
isbn13               72.8%
num_pages            76.2%
publication_year     83.9%
tags                 91.9%
title               100.0%
description         100.0%
average_rating      100.0%
url                 100.0%
image_url           100.0%
dtype: str

## Editions

In [11]:
print(f"book_id  {books.book_id.nunique():,}")
print(f"work_id  {books.work_id.nunique():,}")
print(f"work_id null/empty  {(books.work_id.fillna('') == '').sum():,}")

per_work = books.groupby("work_id").size()
print(f"\nmedian editions/work  {per_work.median():.0f}")
print(f"max                   {per_work.max():,}")
print(f"works with 1 edition  {per_work.eq(1).mean():.1%}")

book_id  792,133
work_id  581,787
work_id null/empty  0

median editions/work  1
max                   373
works with 1 edition  86.7%


In [12]:
books = books.copy()

# Empty strings are NOT skipped by .first() (NaN is), so description presence
# has to be an explicit sort key or a work inherits a blank description from
# its most-rated edition even when a smaller edition has one.
books["_has_desc"] = books.description.fillna("").str.strip().str.len().gt(0)

# work_id is blank on some rows. Without a synthetic key every one of them
# collapses into a single meaningless group.
_blank = books.work_id.fillna("").str.strip().eq("")
books["_work_key"] = books.work_id.where(~_blank, "nowork_" + books.book_id.astype(str))

# Prefer an English edition when one exists.
books["_is_en"] = books.language_code.fillna("").str.strip().str.lower().str.startswith("en")

print(f"editions           {len(books):,}")
print(f"blank work_id      {_blank.sum():,}")
print(f"distinct work keys {books._work_key.nunique():,}")

editions           792,133
blank work_id      0
distinct work keys 581,787


In [14]:
#books.to_parquet(INTERIM / "editions_extra.parquet", index=False)
print(f"{len(books):,} editions -> editions_extra.parquet")

792,133 editions -> editions_extra.parquet


## Canonical edition

**This is the file the later merge needs.** Concatenate `editions.parquet` and
`editions_extra.parquet`, then redo everything from here down — works that have
editions in both files would otherwise be aggregated twice.

In [ ]:
works = (books.sort_values(
             ["_is_en", "_has_desc", "ratings_count"],
             ascending=[False, False, False],
             kind="mergesort")          # stable: ties keep file order, so reruns match
         .groupby("_work_key", as_index=False)
         .first())

print(f"works = {len(works):,}")

works = 581,787


In [ ]:
g = books.groupby("_work_key")

agg = g.agg(
    ratings_total=("ratings_count", "sum"),
    reviews_total=("text_reviews_count", "sum"),
    editions=("book_id", "size"),
    year_first=("publication_year", "min"),
    pages_median=("num_pages", "median"),
    pages_min=("num_pages", "min"),
    pages_max=("num_pages", "max"),
)

# Weighted mean rating. A 5.0 from one rater must not outweigh a 4.1 from
# 200,000, which a flat mean across editions would allow.
_w    = books.ratings_count.fillna(0)
_prod = (books.average_rating.fillna(0) * _w).groupby(books._work_key).sum()
_wsum = _w.groupby(books._work_key).sum()
agg["avg_rating"] = (_prod / _wsum.replace(0, np.nan))

works = works.drop(columns=[c for c in agg.columns if c in works.columns])
works = works.merge(agg, left_on="_work_key", right_index=True, how="left")

_disagree = ((works.pages_max - works.pages_min) > 50).mean()
print(f"pages disagree  {_disagree:.1%} of works by >50pp")

pages disagree  3.3% of works by >50pp


## Merging tags

In [ ]:
def _merge_tags(tags_series, counts_series):
    c = Counter()
    for tags, counts in zip(tags_series, counts_series):
        for t, n in zip(tags, counts):
            c[t] += int(n)
    return c.most_common()

_merged = {
    key: _merge_tags(grp.tags, grp.tag_counts)
    for key, grp in tqdm(books.groupby("_work_key")[["tags", "tag_counts"]])
}

works["tags"]       = works._work_key.map(lambda k: [t for t, _ in _merged[k]])
works["tag_counts"] = works._work_key.map(lambda k: [n for _, n in _merged[k]])

print(works.tags.apply(len).describe())

100%|██████████| 581787/581787 [00:36<00:00, 15880.38it/s]


count    581787.000000
mean         19.676086
std          26.775910
min           0.000000
25%           2.000000
50%           8.000000
75%          23.000000
max         113.000000
Name: tags, dtype: float64


## Genre and IDs

In [ ]:
# A work's editions can carry different genre lists. Union rather than
# inheriting the canonical edition's.
_genres = (books.groupby("_work_key").genre
                .apply(lambda s: sorted({g for lst in s for g in lst})))
works["genres"] = works._work_key.map(_genres)

print(works.genres.apply(len).value_counts().sort_index())

genres
0     235215
1     165833
2      41795
3      42039
4      52206
5      18091
6      11726
7       7695
8       3535
9       2016
10       992
11       467
12       135
13        34
14         6
15         1
16         1
Name: count, dtype: int64


In [ ]:
work_book_ids = (books.groupby("_work_key").book_id
                      .apply(lambda x: sorted(set(x.astype(str)))))

works["book_id_all"] = works._work_key.map(work_book_ids)
works["book_id_all"] = works.book_id_all.apply(lambda v: v if isinstance(v, list) else [])

print(works.book_id_all.apply(len).sum(), "editions accounted for")

792133 editions accounted for


## ID check

In [ ]:
book_to_work = books[["book_id", "_work_key"]].drop_duplicates()

# Every edition's identifiers, for a later external merge. Coverage per work is
# much better than per edition: a work with 40 editions only needs one to carry
# an ASIN.
def _key_list(col):
    s = books[books[col].fillna("").astype(str).str.strip().ne("")]
    return s.groupby("_work_key")[col].apply(lambda x: sorted(set(x.astype(str))))

work_keys = pd.DataFrame({
    "isbn13":      _key_list("isbn13"),
    "isbn":        _key_list("isbn"),
    "asin":        _key_list("asin"),
    "kindle_asin": _key_list("kindle_asin"),
})

for c in work_keys.columns:
    ed = filled(books[c])
    wk = work_keys[c].notna().sum() / len(works)
    print(f"{c:<12} {ed:6.1%} of editions -> {wk:6.1%} of works")

isbn13        72.8% of editions ->  75.8% of works
isbn          61.7% of editions ->  64.2% of works
asin          14.1% of editions ->  17.7% of works
kindle_asin   39.9% of editions ->  40.8% of works


In [ ]:
assert works._work_key.is_unique, "work keys are not unique"
assert len(works) == books._work_key.nunique(), "lost or gained works"

_r_in, _r_out = books.ratings_count.sum(), works.ratings_total.sum()
_t_in  = sum(sum(c) for c in books.tag_counts)
_t_out = sum(sum(c) for c in works.tag_counts)
assert np.isclose(_r_in, _r_out), f"ratings mismatch {_r_in:,} vs {_r_out:,}"
assert _t_in == _t_out, f"tag volume mismatch {_t_in:,} vs {_t_out:,}"

print(f"{len(books):,} editions -> {len(works):,} works")
print(f"ratings preserved     {_r_out:,.0f}")
print(f"tag volume preserved  {_t_out:,}")
print(f"description coverage  {works._has_desc.mean():.1%}")

792,133 editions -> 581,787 works
ratings preserved     234,695,396
tag volume preserved  754,450,402
description coverage  100.0%


In [ ]:
works = works.merge(
    work_keys.rename(columns={"isbn13":      "isbn13_all",
                              "isbn":        "isbn_all",
                              "asin":        "asin_all",
                              "kindle_asin": "kindle_asin_all"}),
    on="_work_key", how="left",
)

# left join yields NaN for any work missing from work_keys; normalize to []
for c in ["isbn13_all", "isbn_all", "asin_all", "kindle_asin_all"]:
    works[c] = works[c].apply(lambda v: v if isinstance(v, list) else [])

assert len(works) == works._work_key.nunique(), "merge duplicated rows"
print(f"{len(works):,} works")

581,787 works


## Language

In [ ]:
lang = works.language_code.fillna("").str.strip().str.lower().replace("", "(missing)")
print(lang.value_counts().head(20).to_string())

def lang_group(c):
    if c == "(missing)":   return "missing"
    if c.startswith("en"): return "english"
    return "other"

groups = lang.apply(lang_group)
print()
print((groups.value_counts(normalize=True) * 100).round(1).to_string())

language_code
(missing)    256446
eng          169477
en-us         20855
spa           15099
ara           14754
ita           13937
en-gb         13673
fre            8355
ind            7816
ger            6754
por            5162
tur            4895
nl             4617
msa            3734
gre            3534
fin            3193
swe            2650
rus            2187
per            1996
bul            1734

language_code
missing    44.1
english    35.3
other      20.6


In [ ]:
DetectorFactory.seed = 0   # langdetect is nondeterministic without this

def detect_lang(text, min_chars=25):
    text = (text or "").strip()
    if len(text) < min_chars:
        return None
    try:
        return detect(text)
    except LangDetectException:
        return None

In [ ]:
blank = works.language_code.fillna("").str.strip().eq("")

known = works[~blank].sample(min(2000, (~blank).sum()), random_state=0).copy()
known["detected"] = known.description.apply(detect_lang)
ok = known.dropna(subset=["detected"])
agree = (ok.detected.str[:2] == ok.language_code.str[:2].str.lower()).mean()
print(f"agreement: {agree:.1%}  (on {len(ok):,} sampled)")

agreement: 73.5%  (on 2,000 sampled)


In [ ]:
tqdm.pandas()

detected = works.loc[blank, "description"].progress_apply(detect_lang)
print(detected.value_counts(dropna=False).head(10).to_string())

100%|██████████| 256446/256446 [11:09<00:00, 382.77it/s]

description
en    246003
es      2377
cy      2323
fr      1151
id       838
de       834
it       434
pt       363
tr       244
nl       238


In [ ]:
works["lang"] = works.language_code.fillna("").str.strip().str[:2].str.lower().replace("", np.nan)
works.loc[blank, "lang"] = detected.str[:2]
works["lang_source"] = np.where(blank, "detected", "goodreads")

print(works.lang.value_counts(dropna=False).head(10).to_string())
print(f"\nstill unknown: {works.lang.isna().mean():.1%}")

lang
en    451326
sp     15099
ar     14755
it     14371
fr      9515
in      7818
po      6802
ge      6756
tu      4900
nl      4857

still unknown: 0.0%


In [ ]:
before = len(works)

keep = works.lang.eq("en")
english = works[keep].reset_index(drop=True)

print(f"{before:,} -> {len(english):,} works  ({keep.mean():.1%} kept)")
print(f"  non-english  {works.lang.notna().sum() - keep.sum():,}")
print(f"  unknown      {works.lang.isna().sum():,}")

581,787 -> 451,326 works  (77.6% kept)
  non-english  130,460
  unknown      1


## Write

In [ ]:
english = english.drop(columns=["work_id"]).rename(columns={"_work_key": "work_id"})
assert english.work_id.is_unique, "work_id not unique after rename"

blank_title = english.title.fillna("").str.strip().eq("")
english = english[~blank_title].reset_index(drop=True)
if "title_without_series" in english.columns:
    english = english.drop(columns=["title_without_series"])

print(f"dropped {blank_title.sum()} blank-title rows -> {len(english):,}")
print(english.columns.tolist())

dropped 1 blank-title rows -> 451,325
['work_id', 'isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'title', 'genre', 'tags', 'tag_counts', 'genres', '_has_desc', '_is_en', 'ratings_total', 'reviews_total', 'editions', 'year_first', 'pages_median', 'pages_min', 'pages_max', 'avg_rating', 'book_id_all', 'isbn13_all', 'isbn_all', 'asin_all', 'kindle_asin_all', 'lang', 'lang_source']


In [27]:
out = INTERIM / "english_works_extra.parquet"
english.to_parquet(out, index=False)

works.to_parquet(INTERIM / "works_wip_extra.parquet", index=False)
book_to_work.to_parquet(INTERIM / "book_to_work_extra.parquet", index=False)

print(f"{len(english):,} rows -> {out}  ({out.stat().st_size / 1e6:.1f} MB)")

check = pd.read_parquet(out)
assert len(check) == len(english), "row count changed"
assert check.work_id.is_unique, "work_id not unique after round-trip"
print("verified")

451,325 rows -> ../data/interim/english_works_extra.parquet  (551.5 MB)
verified


## Merging with the original run

`english_works_extra.parquet` is **not** safe to concatenate with
`english_works.parquet`. A work with editions in both files appears in both
tables, with its ratings, tags and edition counts split between them.

Redo the aggregation on the union of the edition-level files instead:

```python
a = pd.read_parquet(INTERIM / "editions.parquet")
b = pd.read_parquet(INTERIM / "editions_extra.parquet")
books = pd.concat([a, b], ignore_index=True).drop_duplicates("book_id")
```

Then run this notebook from **Canonical edition** down on that frame.
Language detection can be skipped for rows that already carry a resolved
`lang`, which is most of them.

In [28]:
a = pd.read_parquet(INTERIM / "editions.parquet")
b = pd.read_parquet(INTERIM / "editions_extra.parquet")

overlap = set(a.work_id) & set(b.work_id)
print(f"editions   {len(a):,} + {len(b):,}")
print(f"overlapping work_ids: {len(overlap):,}")

editions   1,244,611 + 792,133
overlapping work_ids: 61


In [29]:
a = pd.read_parquet(INTERIM / "english_works.parquet")
b = pd.read_parquet(INTERIM / "english_works_extra.parquet")

works = pd.concat([a, b], ignore_index=True)
dupes = works.work_id.duplicated(keep=False).sum()
print(f"{len(works):,} works, {dupes} duplicate rows")

works = works.sort_values("ratings_total", ascending=False) \
             .drop_duplicates("work_id", keep="first") \
             .reset_index(drop=True)
print(f"{len(works):,} after dedup")

1,002,652 works, 90 duplicate rows
1,002,607 after dedup


In [30]:
for q in ["Dorian Gray", "One Hundred Years of Solitude", "Ivan Ily", "Little Prince"]:
    hits = works[works.title.str.contains(q, case=False, na=False, regex=False)]
    if hits.empty:
        print(f"{q}: NO MATCH")
    else:
        r = hits.nlargest(1, "ratings_total").iloc[0]
        print(f"{q}: {r.title} ({r.ratings_total:,})")

Dorian Gray: The Picture of Dorian Gray (683,950)
One Hundred Years of Solitude: One Hundred Years of Solitude (578,926)
Ivan Ily: The Death of Ivan Ilych (55,051)
Little Prince: The Little Prince (878,752)
